In [9]:
#if fresh enviornment, in terminal
'''
chmod +x dependencies.sh
./dependencies.sh
'''
import sys
print(sys.executable)

/home/ec2-user/anaconda3/envs/python3/bin/python


In [10]:
#load dependacies
!pip install -q transformers datasets

Using device: cuda


In [20]:
!pip install seaborn

In [27]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
import re, os, math
import seaborn as sns
from collections import Counter
import itertools

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [23]:
#load data into df
#load the json training_set_task1.txt
with open("training_set_task1.txt", "r", encoding="utf-8") as f:
    data = json.load(f)

#load into pandas DataFrame
df = pd.DataFrame(data)

# Count using pandas (if 'labels' is a column containing lists)
empty_label_count = df["labels"].apply(lambda x: len(x) == 0).sum()

print(f"Number of entries with empty labels: {empty_label_count}")

#save entries with empty labels into a seperate DF for checking
empty_labels_df = df[df["labels"].map(len) == 0].copy()

# Keep only valid non-empty entries in your primary DataFrame
df = df[df["labels"].map(len) > 0].copy()

# Verify the row counts
print(f"Valid entries saved to df: {len(df)}")
print(f"Empty label entries saved to empty_labels_df: {len(empty_labels_df)}")


Number of entries with empty labels: 143
Valid entries saved to df: 545
Empty label entries saved to empty_labels_df: 143


In [33]:
# Class weights for imbalanced data

# 1. Flatten all labels
all_labels = list(itertools.chain.from_iterable(df["labels"]))

# 2. Extract unique labels and sort them to ensure consistent ordering
label_names = sorted(list(set(all_labels)))
NUM_CLASSES = len(label_names)

# 3. Count occurrences of each class 
class_counts = Counter(all_labels)

# 4. Compute inverse frequency weights
total_occurrences = len(all_labels)
# Note: We now iterate through 'label_names' instead of integer indices
class_weights = torch.tensor(
    [total_occurrences / max(1, class_counts.get(name, 0)) for name in label_names],
    dtype=torch.float32
).to(device)

# 5. Normalize weights so they average to 1.0
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

# Output formatted results
print('Class weights (inverse frequency):')
for i, name in enumerate(label_names):
    print(f'  {name}: {class_weights[i].item():.2f}')


Class weights (inverse frequency):
  Appeal to authority: 0.63
  Appeal to fear/prejudice: 0.19
  Bandwagon: 4.09
  Black-and-white Fallacy/Dictatorship: 0.45
  Causal Oversimplification: 0.30
  Doubt: 0.17
  Exaggeration/Minimisation: 0.16
  Flag-waving: 0.30
  Glittering generalities (Virtue): 0.26
  Loaded Language: 0.02
  Misrepresentation of Someone's Position (Straw Man): 0.41
  Name calling/Labeling: 0.04
  Obfuscation, Intentional vagueness, Confusion: 2.04
  Presenting Irrelevant Data (Red Herring): 8.17
  Reductio ad hitlerum: 0.91
  Repetition: 1.02
  Slogans: 0.19
  Smears: 0.04
  Thought-terminating cliché: 0.41
  Whataboutism: 0.20


### There are 2 labels missing from your results:
    * Transfer
    * Appeal to (Strong) Emotions

In [34]:
#print off a few of the ones that had no labels
empty_labels_df.head()

,id,labels,text
1,189,[],This is not an accident!
4,15,[],WHO TRUMP REPRESENTS\n\nWHO DEMOCRATS REPRESENT\n
22,142,[],DEMOCRATS STARTED WEARING ALL WHITE...\n\nAGAIN
25,176,[],CORRECT:\n\nTHE WORLD\n\nCHINA\n
28,178,[],COVID-19 is everywhere but countries with head...


word order carries meaning. Important